# Description du projet

Ce projet a pour ambition de concevoir un agent artificiel capable d'apprendre à jouer à Onitama, un jeu de stratégie abstraite à deux joueurs. L'agent repose sur un réseau de neurones profond entraîné sur base d'expériences de jeu puis d'entraînement contre lui-même.
L'objectif final est d'obtenir un agent compétitif, capable de rivaliser avec un joueur humain averti, en ayant développé par lui-même des stratégies cohérentes et efficaces propres à la complexité tactique d'Onitama.

# Description du jeu Onitama

Onitama est un jeu de stratégie abstraite à deux joueurs, de Shimpei Sato, publié par Arcane Wonders en 2014. Si le jeu est sobre en termes de son aspect et de ses règles, il est très riche en termes de profondeur tactique. Le jeu se joue sur un plateau de 5×5, chaque joueur dispose de 5 pièces : le Maître, qui est le pion principal du joueur et qui est placé au centre de la ligne de départ du joueur.4 pièces d’Élèves, qui sont placées près de leur Maître, à droite et à gauche sur la même ligne. Les joueurs sont situés aux côtés opposés du plateau plateau nord et du plateau sud.

La partie débute avec 5 cartes piochées aléatoirement parmi les 16 disponibles. Chaque joueur reçoit 2 cartes, une cinquième, la carte "neutre", détermine le premier joueur et est placée à droite du plateau. Chaque carte se caractérise par un diagramme de mouvement qui tire son inspiration des mouvements des arts martiaux japonais (Dragon, Cobra, Eel et d’autres), représentant les mouvements possibles à partir de la position actuelle du pion à déplacer. Il y a deux cartes dans la main de chaque joueur et la cinquième est mise à côté de l’échiquier, prête à l’emploi.

À son tour, le joueur doit obligatoirement :
- Choisir l'une de ses deux cartes pour effectuer un déplacement.
- Déplacer l'une de ses pièces (Maître ou Élève) selon le schéma indiqué par la carte choisie.
- Échanger la carte utilisée avec la carte en attente placée sur le côté — qui devient alors disponible pour son adversaire au prochain tour.

Ce mécanisme d'échange crée une symétrie et une tension constante : les cartes circulent entre les deux joueurs tout au long de la partie, rendant chaque décision doublement stratégique (bouger efficacement et ne pas offrir un trop grand avantage à l'adversaire).

Il existe deux façons de remporter la partie :
- La Voie de la Pierre : capturer le Maître adverse en déplaçant l'une de ses pièces sur sa case.
- La Voie du Courant : amener son propre Maître sur le Temple adverse — la case centrale de la rangée de départ de l'ennemi.

# Objectifs

Voici les objectifs que je me suis fixés dans ce projet d'apprentissage des techniques d'apprentissage profond et d'apprentissage par renforcement :

1. Objectif "or"
Développer un joueur basé sur un réseau de neurones profonds, avec ou sans complément d'algorithmes complémentaires, étant de mesure de battre tous les adversaires heuristiques ayant servi à son entraînement. Pour ce faire le réseau doit avoir compris les règles du jeu et développer des stratégies avancées. 

2. Objectif "argent"
Développer un joueur basé sur un réseau de neurones profonds, avec ou sans complément d'algorithmes complémentaires, étant de mesure de battre des adversaires de type heuristiques de niveau moyen. Pour ce faire le réseau doit avoir compris les règles du jeu et développer des stratégies limitées.

3. Objectif "bronze"
Développer un joueur basé sur un réseau de neurones profonds, avec ou sans complément d'algorithmes complémentaires, étant de mesure de battre des adversaires de type heuristiques de niveau faible. Pour ce faire le réseau doit avoir compris les règles du jeu générales et savoir jouer "simplement".



# Contraintes

Le projet étant réalisé dans le cadre de la formation Alyra, les contraintes suivantes sont à considérer :
- Ressources matérielles limitées : les objectifs devront être atteints en utilisant les performances matérielles offertes par le matériel à ma disposition, c'est à dire un entraînement sur CPU uniquement. 
- les modèles développés pour atteindre les objectifs devront se baser sur des réseaux de neurones profonds, même si des algorithmes complémentaires pourront être utiliser pour renforcer leur efficacité. (Pas d'utilisation d'heuristique dans ces modèles)
- contraintes temporelles de finalisation

# Résumé de la démarche globale

La première étape du projet a consisté à définir puis à matérialiser l'environnement de jeu, c'est à dire un cadre global permettant de jouer une ou des parties avec des joueurs "humains" ou "synthétiques".

Cette première étape réalisée j'ai conçu et développé un ensemble de joueurs "synthétiques" de référence, ceci afin de répondre à deux objectifs : 
- Permettre de générer des données de parties en grand nombre en faisant jouer les joueurs "synthétiques" les uns contre les autres.
- Permettre l'évalution des modèles qui seront réalisés face à des adversaires de différents niveaux.

Différents modèles basés sur des réseaux de neurones, sur base de couches convolutives ou denses et différentes architectures ont été implémentés (Cf. notebook versions-modeles.ipynb). L'ensemble de ces modèles possède une couche d'entrée identique (pour compatibilité avec les données d'entraînement) et deux têtes :
- Policies head : prédiction de politique, ou autrement dit "Dans la situation actuelle, quelles sont les meilleures actions"
- Value head : prédiction de valeur, ou autrement dit "Est ce que la situation actuelle est une bonne ou une mauvaise situation"

L'idée est ensuite d'entraîner les modèles en plusieurs étapes successives : 
- Entraînement supervisé
- Entraînement supervisé de la tête de valeur
- Entraînement PPO

L'évaluation des modèles obtenus permettra de déterminer les capacités des modèles "purs" à atteindre les objectifs fixés et donc leur capacités à résoudre les problématiques de jeu de manière "native", c'est à dire purement via "l'intuition" du réseau.

Ces modèles pourront enfin être mis en combinaison avec d'autres algorithmes de manière à optimiser leurs performances et obtenir des joueurs "mixtes". Ces joueurs seront également évalués afin de déterminer leurs capacités à atteindre les objectifs fixés. 

Ces différentes étapes de conception, d'entraînement et de validation seront détaillées dans les chapitres suivants. 

# Environnement de jeu

### Définitions

**L'environnement général du jeu est déterminé par :**
- Un **état** actuel
- Le joueur courant (rouge ou bleu)
- La correspondance entre le les joueurs réels et leurs couleurs, ordre de jeu
- Les 16 **cartes** disponibles dans le jeu

**Une carte est déterminée par :**
- Un identifiant numérique unique (de 0 à 15)
- Un nom
- Une couleur (déterminant le premier joueur si c'est carte est la première carte neutre du jeu)
- N **mouvements** relatifs

**Un mouvement d'une carte est déterminé par :**
- Un déplacement relatif selon l'axe x
- Un déplacement relatif selon l'axe y
- Un identifiant unique (de 0 à 51, soit 52 mouvements possibles)

**Un état (C'est à dire l'état du jeu à l'instant T) est déterminé par :**
- Une grille de jeu de 5x5 sur laquelle sont placés : 
    - 4 étudiants du joueur courant
    - 1 maître du jour courant
    - 4 étudiants de l'opposant
    - 1 maître de l'opposant
- 2 cartes disponibles pour le joueur courant
- 2 cartes disponibles pour l'opposant
- 1 carte neutre
Il est important de noter que l'état est toujours représenté du point du vue du joueur courant, le plateau fait donc l'objet d'une translation à chaque coup joué, ceci afin de permettre une meilleure utilisabilité des historiques d'état et exploitation par le réseau.

### Implémentation concrète
Les modules suivants, présents dans le répertoire "onitama/" mettent en oeuvre l'implémentation de l'environnement : 
- game.py   :   Mise en oeuvre d'une partie (classe Game) ou d'un ensemble de N parties (classe GameSession)
- board.py  :   Représentation de l'état actuel (classe Board) et ensemble de fonctions permettant de récupérer les coups possibles, de les jouer et de les annuler
- card.py   :   Implémentation des définitions des cartes et mouvements
- constants :   Constantes générales utilisées dans les différentes classes relatives aux environnements de jeu

### Représentation de l'état pour le réseau

Un état sera représenté sous la forme d'une matrice de 5x5 sur 10 plans (5x5x10) :
- Plan 0 : positions des pions du joueurs courants (0 ou 1)
- Plan 1 : positions des pions de l'adversaire (0 ou 1)
- Plan 2 : position du maître du joueur courant (0 ou 1)
- Plan 3 : position du maître de l'adversaire (0 ou 1)
- Plan 4 : Mouvements de la carte 1 du joueur courant (considérant la position de la pièce en 2:2, 1 pour les destinations relatives possibles)
- Plan 5 : Mouvements de la carte 2 du joueur courant (considérant la position de la pièce en 2:2, 1 pour les destinations relatives possibles)
- Plan 6 : Mouvements de la carte 1 de l'adversaire (considérant la position de la pièce en 2:2, 1 pour les destinations relatives possibles)
- Plan 7 : Mouvements de la carte 2 de l'adversaire (considérant la position de la pièce en 2:2, 1 pour les destinations relatives possibles)
- Plan 8 : Mouvements de la carte neutre (considérant la position de la pièce en 2:2, 1 pour les destinations relatives possibles)
- Plan 9 : Joueur courant premier joueur ou non (Grille de 0 ou de 1)

Cet encodage de l'état a été choisi afin de permettre une représentation "géographique" facilitée des positions, en particulier pour les réseaux à base de couches convolutives. 

# Joueurs de référence

# Génération des données de jeu

# Evaluation des joueurs de référence

# Architectures et versions de modèles

# Entraînement supervisé

# Entraînement supervisé de la tête de valeur

# Entraînement PPO

# Evaluation des modèles "purs"

# Joueurs "mixtes" (Modèle + algorithme)

# Evaluation des joueurs "mixtes"

# Conclusion